In [ ]:
#%pip install pip --upgrade
#%pip install --upgrade pip setuptools wheel

In [ ]:
#%pip install langchain openai neo4j langchain-openai textdistance pandas spacy\
#    langchain-community seaborn openpyxl chromadb markdown bs4 pypdf neo4j-graphrag\
#        https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

#!/usr/local/bin/python3.13 -m spacy download en_core_web_md
#!python3 -m spacy download en_core_web_md

In [ ]:
import os
import shutil
import json
import re
import pandas as pd
import tiktoken
import math
import spacy
from typing import Set, Any, Union, Dict, List, Tuple, Hashable
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import chromadb
import textdistance
from tqdm.auto import tqdm
from neo4j.exceptions import Neo4jError, ClientError

# On MAC:
!export HNSWLIB_NO_NATIVE=1

tqdm.pandas()

In [ ]:
# Set model name
model_name="gpt-4o" # "gpt-4o-mini", IDs of OpenAI's FT models "ft:gpt-4o-2024-08-06:..."

DATA_PATH = "./NLquestions"
CYPHER_PATH = "./CypherQueries"
NEO4J_OUTPUTS_PATH = "./Neo4jOutputs"

from neo4j import GraphDatabase

class HetionetGraph:
    def __init__(self, uri="bolt://neo4j.het.io", username="neo4j", password="neo4j"):
        self._driver = GraphDatabase.driver(uri, auth=(username, password))
        self.schema = ""

    def query(self, query, params=None):
        with self._driver.session() as session:
            result = session.run(query, params or {})
            return [r.data() for r in result]

    def refresh_schema(self):
        try:
            labels = self.query("CALL db.labels()")
            rels = self.query("CALL db.relationshipTypes()")
            props = self.query("CALL db.propertyKeys()")

            labels_txt = ", ".join(sorted(x["label"] for x in labels if "label" in x))
            rels_txt = ", ".join(sorted(x["relationshipType"] for x in rels if "relationshipType" in x))
            props_txt = ", ".join(sorted(x["propertyKey"] for x in props if "propertyKey" in x))

            self.schema = (
                f"Node labels: {labels_txt}\n"
                f"Relationship types: {rels_txt}\n"
                f"Property keys: {props_txt}"
            )
        except Exception as e:
            self.schema = f"Schema unavailable: {e}"

        return self.schema

    def close(self):
        self._driver.close()

graph = HetionetGraph()
graph.refresh_schema()
print(graph.schema)

***
# Creating the dataset

In [ ]:
import json
import pandas as pd

def load_level(path, level_name="all"):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data).reset_index(drop=True)
    df["ID"] = f"{level_name}/question_" + (df.index + 1).astype(str)
    df["level"] = level_name
    return df

df_all = load_level("FTdataset.json")

# The following is useful to check the length of the longest cypher query and limit the max number of tokens accordingly (this speed ups evaluation)
print("Longest cypher:", df_all["cypher"].str.len().max())
longest = df_all[df_all["cypher"].str.len() == df_all["cypher"].str.len().max()]['cypher']
print(longest)
encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(longest.iloc[0])
MAX_TOKENS = math.ceil(len(tokens) / 100) * 100
print(len(tokens), MAX_TOKENS)

test_df  = df_all.groupby("level", group_keys=False).sample(frac=0.10, random_state=42)
train_df = df_all.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_df[['question', 'cypher']].to_json("FTdataset_local.json", orient="records", indent=2)

SYSTEM_PROMPT = (
    "Given an input question, convert it to a Cypher query. No pre-amble."
)

def row_to_chat_example(row, add_system=True):
    messages = []
    if add_system:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": row["question"]})
    messages.append({"role": "assistant", "content": row["cypher"]})
    return {"messages": messages}

out_path = "FTdataset_GPTviaGUI.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        ex = row_to_chat_example(row, add_system=True) 
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_df.head(n=3)

In [ ]:
test_df.head(n=3)

***
# Pre-processing for building a vector store for performing RAG
## Step-by-step tutorial for Creating a Vector Store with ChromaDB

In [ ]:
def build_skip_set(test_df):
    skip = set()
    for qid in test_df["ID"]:
        skip.add(f"{qid}.txt")
    return skip

skip_files = build_skip_set(test_df)

def load_txt_documents():
    """
    Load .txt documents from the specified directory.
    Returns: List of Document objects
    """
    all_documents = []

    for folder_name in os.listdir(DATA_PATH):
        folder_path = os.path.join(DATA_PATH, folder_name)
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                if file_name.endswith(".txt") and folder_name + "/" + file_name not in skip_files:
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text_content = f.read()
                            all_documents.append(Document(
                                page_content=text_content,
                                metadata={"type": "text", "source": folder_name + "/" + file_name}
                            ))
                    except Exception as e:
                        print(f"Error loading TXT file {file_name}: {e}")

    return all_documents

Check if it works.

In [ ]:
documents = load_txt_documents()

bad = [
    d.metadata["source"]
    for d in documents
    if os.path.basename(d.metadata["source"]) in skip_files
]

assert len(bad) == 0, f"LEAKAGE! Indexed test files: {bad[:5]}"

In [ ]:
documents = load_txt_documents()

documents[1].page_content, documents[1].metadata['source']

***

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = "...YOUR_OPENAI_API_KEY_HERE..." 
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# Re-create collection if it previously existed
try:
    chroma_client.delete_collection("my_collection")
except:
    pass

collection = chroma_client.create_collection(name="my_collection")
texts = [doc.page_content for doc in documents]
ids = [doc.metadata["source"] for doc in documents]
metas = [doc.metadata for doc in documents]

BATCH = 512

for start in range(0, len(texts), BATCH):
    batch_texts = texts[start:start+BATCH]
    batch_ids   = ids[start:start+BATCH]
    batch_metas = metas[start:start+BATCH]

    resp = client.embeddings.create(
        model="text-embedding-3-large",
        input=batch_texts
    )
    batch_embeddings = [item.embedding for item in resp.data]

    collection.add(
        documents=batch_texts,
        ids=batch_ids,
        metadatas=batch_metas,
        embeddings=batch_embeddings
    )

In [ ]:
# Check if it works
collection = chroma_client.get_collection(name="my_collection")
query = "Find all gene names containing 'PIK3CA'"

query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i, doc in enumerate(results["documents"][0]):
    print(f"Match {i+1}:")
    print("→", doc)
    print("Filename:", results["metadatas"][0][i]["source"])
    print("Distance:", results["distances"][0][i])
    print("-" * 40)

***
# Metrics

We will use the following metrics to evaluate the Cypher generation ability of LLMs.
* **Jaro-Winkler**
* **Jaccard**
* **Coverage**
* **Pass**:

In [ ]:
def get_jw_distance(string1: str, string2: str) -> float:
    """
    Calculate the Jaro-Winkler distance between two strings.

    The Jaro-Winkler distance is a measure of similarity between two strings.
    The score is normalized such that 0 equates to no similarity and
    1 is an exact match.
    """
    # Call the jaro_winkler function from the textdistance library.
    return textdistance.jaro_winkler(string1, string2)

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple

from typing import Any, Hashable

from collections.abc import Mapping, Iterable
from typing import Any, Hashable

def _norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    if isinstance(v, Mapping):
        return frozenset(_norm_value(x) for x in v.values())

    if isinstance(v, (list, tuple)):
        return frozenset(_norm_value(x) for x in v)

    if isinstance(v, set):
        return frozenset(_norm_value(x) for x in v)

    return v if isinstance(v, Hashable) else str(v)


def rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[Set], List[Set]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [{_norm_value(v) for v in row.values()} for row in dictL]
    setViewsR = [{_norm_value(v) for v in row.values()} for row in dictR]
    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = rowsim(setViewsL[i], setViewsR[j])
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL
    return setViewsL, setViewsR

def df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        view_L = [[_norm_value(v) for v in row.values()] for row in dictL]
        view_R = [[_norm_value(v) for v in row.values()] for row in dictR]
    else:
        view_L, view_R = make_alignment(dictL, dictR)

    totalL = {(i, x) for i, row in enumerate(view_L) for x in set(row)}
    totalR = {(i, x) for i, row in enumerate(view_R) for x in set(row)}

    union = totalL | totalR
    return 1.0 if not union else len(totalL & totalR) / len(union)

def jaccard_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    return df_sim(dict_L, dict_R, "order by" in f"{cypher_L}".lower())

In [ ]:
from typing import Any, Dict, Hashable, List, Set, Tuple
from collections.abc import Mapping

def t2c_norm_value(v: Any) -> Hashable:
    if isinstance(v, str):
        s = v.strip()
        try:
            return float(s)
        except ValueError:
            return s.lower()

    if isinstance(v, (int, float)):
        return float(v)

    return v if isinstance(v, Hashable) else str(v)

def t2c_flat_values(v: Any) -> Set[Hashable]:
    vals: Set[Hashable] = set()

    if isinstance(v, (str, int, float)):
        vals.add(t2c_norm_value(v))
        return vals

    if isinstance(v, Mapping):
        for x in v.values():
            vals |= t2c_flat_values(x)
        return vals

    if isinstance(v, (list, tuple, set)):
        for x in v:
            vals |= t2c_flat_values(x)
        return vals

    vals.add(t2c_norm_value(v))
    return vals


def t2c_row_values(row: Dict) -> Set[Hashable]:
    s: Set[Hashable] = set()
    for v in row.values():
        s |= t2c_flat_values(v)
    return s

def t2c_rowsim(setL: Set[Hashable], setR: Set[Hashable]) -> float:
    union = setL | setR
    return 1.0 if not union else len(setL & setR) / len(union)

def t2c_make_alignment(dictL: List[Dict], dictR: List[Dict]) -> Tuple[List[List[Hashable]], List[List[Hashable]]]:
    swap = len(dictL) > len(dictR)

    setViewsL = [list(t2c_row_values(row)) for row in dictL]
    setViewsR = [list(t2c_row_values(row)) for row in dictR]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    for i in range(len(setViewsL)):
        max_sim, max_j = -1.0, i
        for j in range(i, len(setViewsR)):
            sim = t2c_rowsim(set(setViewsL[i]), set(setViewsR[j]))
            if sim > max_sim:
                max_sim, max_j = sim, j
        setViewsR[i], setViewsR[max_j] = setViewsR[max_j], setViewsR[i]

    if swap:
        setViewsL, setViewsR = setViewsR, setViewsL

    return setViewsL, setViewsR

def t2c_row_attr_recall(gt_row: Dict, pred_values: Set[Hashable]) -> float:
    num_attrs = len(gt_row)
    if num_attrs == 0:
        return 1.0

    matched = 0
    for v in gt_row.values():
        gt_vals = t2c_flat_values(v)
        if gt_vals & pred_values:
            matched += 1

    return matched / num_attrs

def t2c_df_sim(dictL: List[Dict], dictR: List[Dict], order_sensitive: bool) -> float:
    if order_sensitive:
        lenL, lenR = len(dictL), len(dictR)
        N = max(lenL, lenR)

        if N == 0:
            return 1.0 

        row_scores: List[float] = []

        for i in range(N):
            if i < lenL:
                gt_row = dictL[i]
                pred_vals = t2c_row_values(dictR[i]) if i < lenR else set()
                row_scores.append(t2c_row_attr_recall(gt_row, pred_vals))
            else:
                pred_vals = t2c_row_values(dictR[i])
                row_scores.append(0.0 if pred_vals else 1.0)

        return sum(row_scores) / N

    view_L, view_R = t2c_make_alignment(dictL, dictR)

    total_attrs = 0
    matched_attrs = 0

    for i, gt_row in enumerate(dictL):
        total_attrs += len(gt_row)
        pred_vals = set(view_R[i]) if i < len(view_R) else set()
        row_recall = t2c_row_attr_recall(gt_row, pred_vals)
        matched_attrs += row_recall * len(gt_row)

    if total_attrs == 0:
        return 1.0

    return matched_attrs / total_attrs

def coverage_df_sim_pair(pair_L, pair_R) -> float:
    cypher_L, dict_L = pair_L
    cypher_R, dict_R = pair_R
    order_sensitive = "order by" in f"{cypher_L}".lower()
    return max(t2c_df_sim(dict_L, dict_R, order_sensitive), df_sim(dict_L, dict_R, order_sensitive))


In [ ]:
# Example usage
dict_a = [{"id": 2, "ReturnedValue": "B"}, {"id": 1, "value": "A"}]
dict_b = [{"id": 1, "value": "A"}, {"id": "2", "value": "B", "extraValue": "C"}, {"id": 1, "value": "A"}]

query_a = "MATCH (n:TableA) RETURN n.id AS id, n.value AS value"
query_b = "MATCH (n:TableB) RETURN n.id AS id, n.value AS value"

similarity_score = jaccard_df_sim_pair((query_a, dict_a), (query_b, dict_b))
print(similarity_score)
similarity_score = coverage_df_sim_pair((query_a, dict_a), (query_b, dict_b))
print(similarity_score)

***
## Generating Cypher statements

***
# Vanilla

Here, we use the LLM alone to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
llm = ChatOpenAI(model_name=model_name, temperature=0)
model_name = "gpt-4o" # "gpt-4o-mini", "FTgpt-4o", , "FTgpt-4o-mini"

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a Cypher query. No pre-amble.",
        ),
        ("human", cypher_template),
    ]
)

cypher_chain = ( cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)


In [ ]:
response = cypher_chain.invoke(
    {
        "question": "Which genes participate in the 'DNA metabolic process' and 'U1 snRNA binding'?"
    }
)
print(response)

In [ ]:
def evaluate_cypher_queries(df, graph, cypher_chain, cypher_prompt, k=1)
    """
    Evaluate cypher queries and update the DataFrame with new evaluation metrics.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least the columns "question" and "cypher".
        graph: Neo4jGraph object used to run queries.
        cypher_chain: LLM chain for generating Cypher queries.
        cypher_prompt: Prompt template for generating Cypher queries.
        k (int): Number of generated Cypher queries to evaluate per question (default is 1).

    Returns:
        pd.DataFrame: The updated DataFrame with new columns:
        "generated_cypher", "true_data", "eval_data", "jaro_winkler", "pass_1", "pass_3", "jaccard".
    """
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    normalized_levenshteins = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Get the true data based on the test Cypher statement.
        true_data = graph.query(row["cypher"])
        
        # Generate cypher queries and fetch data.
        example_generated_cyphers = []
        example_eval_datas = []
        prompts = []
        for _ in range(k):
            cypher = cypher_chain.invoke({"question": row["question"]})
            formatted_prompt = cypher_prompt.format_messages(question=row["question"])
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })
            example_generated_cyphers.append(cypher)
            try:
                example_eval_datas.append(graph.query(cypher))

            except ClientError as e:
                if e.code == "Neo.ClientError.Statement.SyntaxError":
                    print("Cypher SyntaxError:", e)
                    example_eval_datas.append([{"id": "Cypher syntax error"}])

                elif e.code == "Neo.ClientError.Statement.SemanticError":
                    print("Cypher SemanticError:", e)
                    example_eval_datas.append([{"id": "Cypher semantic error"}])

                else:
                    print("Cypher ClientError:", e.code, e)
                    example_eval_datas.append([{"id": "Cypher client error"}])

            except Neo4jError as e:
                print("Neo4j runtime error:", e)
                example_eval_datas.append([{"id": "Neo4j runtime error"}])

            except Exception as e:
                print("Other error:", type(e), e)
                example_eval_datas.append([{"id": "Other error"}])

        # Compute metrics using the first generated cypher/response.
        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        passs = 1 if coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        ) == 1 else 0

        # Append computed values.
        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        
        prompt.append(prompts)

    # Add evaluated results as new columns to the DataFrame.
    df["generated_cypher"] = generated_cyphers
    df["true_data"] = true_datas
    df["eval_data"] = eval_datas
    df["jaro_winkler"] = jaro_winklers
    
    df["jaccard"] = jaccards
    df["coverage"] = coverages
    df["pass"] = passes
    
    df["prompt"] = prompt

    return df

In [ ]:
df = evaluate_cypher_queries(test_df, graph, cypher_chain, cypher_prompt)
df.head(n=3)

In [ ]:
row = df.iloc[2]
# Print the desired information
print("Question:", row["question"], "\n")
print("True Cypher:", row["cypher"], "\n")
print("Generated Cypher:", row["generated_cypher"][0], "\n")

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_vanilla.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_vanilla.pkl")
df.head(n=3)

***
# Schema

As described in [The Impact of Schema Representation in the Text2Cypher Task](https://neo4j.com/blog/developer/schema-representation-in-text2cypher/) ([paper](https://doi.org/10.48550/arXiv.2505.05118)), the base schema above contains nodes, relationships, and their properties. The enhanced schema instead includes example data values and types.

In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional
from neo4j import GraphDatabase

LIST_LIMIT = 128


def _clean_string_values(text: str) -> str:
    return str(text).replace("\n", " ").replace("\r", " ")


def _value_sanitize(d: Any) -> Any:
    if isinstance(d, dict):
        new_dict = {}
        for key, value in d.items():
            if isinstance(value, dict):
                sanitized_value = _value_sanitize(value)
                if sanitized_value is not None:
                    new_dict[key] = sanitized_value
            elif isinstance(value, list):
                if len(value) < LIST_LIMIT:
                    sanitized_value = _value_sanitize(value)
                    if sanitized_value is not None:
                        new_dict[key] = sanitized_value
            else:
                new_dict[key] = value
        return new_dict
    elif isinstance(d, list):
        if len(d) < LIST_LIMIT:
            return [_value_sanitize(item) for item in d if _value_sanitize(item) is not None]
        return None
    return d


def query_database(
    driver,
    query: str,
    params: Optional[Dict[str, Any]] = None,
    sanitize: bool = False,
) -> List[Dict[str, Any]]:
    with driver.session() as session:
        result = session.run(query, params or {})
        json_data = [r.data() for r in result]
        if sanitize:
            json_data = [_value_sanitize(el) for el in json_data]
        return json_data


def infer_type_from_examples(values):
    vals = [v for v in values if v is not None]
    if not vals:
        return "UNKNOWN"

    if any(isinstance(v, list) for v in vals):
        return "LIST"

    if all(isinstance(v, bool) for v in vals):
        return "BOOLEAN"

    def is_int(x):
        try:
            if isinstance(x, bool):
                return False
            if isinstance(x, int):
                return True
            if isinstance(x, float):
                return x.is_integer()
            s = str(x)
            return s.isdigit() or (s.startswith("-") and s[1:].isdigit())
        except Exception:
            return False

    def is_float(x):
        try:
            if isinstance(x, bool):
                return False
            float(x)
            return True
        except Exception:
            return False

    if all(is_int(v) for v in vals):
        return "INTEGER"
    if all(is_float(v) for v in vals):
        return "FLOAT"
    return "STRING"

def summarize_list_values(raw_vals):
    lists = [v for v in raw_vals if isinstance(v, list)]
    if not lists:
        return {
            "values": [],
            "item_examples": [],
            "min_size": None,
            "max_size": None,
        }

    sizes = [len(v) for v in lists]

    item_examples = []
    for lst in lists:
        for item in lst:
            if item is not None:
                item_examples.append(_clean_string_values(item))
            if len(item_examples) >= 3:
                break
        if len(item_examples) >= 3:
            break

    list_examples = []
    for lst in lists[:3]:
        list_examples.append(str(lst[:5]))

    return {
        "values": list_examples,
        "item_examples": item_examples[:3],
        "min_size": min(sizes) if sizes else None,
        "max_size": max(sizes) if sizes else None,
    }


def _get_node_labels(driver) -> List[str]:
    rows = query_database(driver, "CALL db.labels()")
    return [r["label"] for r in rows if "label" in r]


def _get_relationship_types(driver) -> List[str]:
    rows = query_database(driver, "CALL db.relationshipTypes()")
    return [r["relationshipType"] for r in rows if "relationshipType" in r]


def _get_node_properties(driver, label: str, sample: int):
    q = f"""
    MATCH (n:`{label}`)
    WITH n LIMIT {sample}
    UNWIND keys(n) AS k
    WITH k, collect(n[k])[0..10] AS vals
    RETURN k AS property, vals
    ORDER BY property
    """
    rows = query_database(driver, q)

    props = []
    for row in rows:
        raw_vals = row.get("vals", [])
        prop_type = infer_type_from_examples(raw_vals)

        if prop_type == "LIST":
            list_info = summarize_list_values(raw_vals)
            prop_info = {
                "property": row["property"],
                "type": "LIST",
                "values": list_info["values"],
                "item_examples": list_info["item_examples"],
                "min_size": list_info["min_size"],
                "max_size": list_info["max_size"],
            }
        else:
            examples = [_clean_string_values(v) for v in raw_vals[:3] if v is not None]
            prop_info = {
                "property": row["property"],
                "type": prop_type,
                "values": examples,
            }

            if prop_type in ["INTEGER", "FLOAT"]:
                numeric_vals = []
                for v in raw_vals:
                    try:
                        numeric_vals.append(float(v))
                    except Exception:
                        pass
                if numeric_vals:
                    prop_info["min"] = min(numeric_vals)
                    prop_info["max"] = max(numeric_vals)

        props.append(prop_info)

    return props


def _get_rel_properties(driver, rel_type: str, sample: int):
    q = f"""
    MATCH ()-[r:`{rel_type}`]->()
    WITH r LIMIT {sample}
    UNWIND keys(r) AS k
    WITH k, collect(r[k])[0..10] AS vals
    RETURN k AS property, vals
    ORDER BY property
    """
    rows = query_database(driver, q)

    props = []
    for row in rows:
        raw_vals = row.get("vals", [])
        prop_type = infer_type_from_examples(raw_vals)

        if prop_type == "LIST":
            list_info = summarize_list_values(raw_vals)
            prop_info = {
                "property": row["property"],
                "type": "LIST",
                "values": list_info["values"],
                "item_examples": list_info["item_examples"],
                "min_size": list_info["min_size"],
                "max_size": list_info["max_size"],
            }
        else:
            examples = [_clean_string_values(v) for v in raw_vals[:3] if v is not None]
            prop_info = {
                "property": row["property"],
                "type": prop_type,
                "values": examples,
            }

            if prop_type in ["INTEGER", "FLOAT"]:
                numeric_vals = []
                for v in raw_vals:
                    try:
                        numeric_vals.append(float(v))
                    except Exception:
                        pass
                if numeric_vals:
                    prop_info["min"] = min(numeric_vals)
                    prop_info["max"] = max(numeric_vals)

        props.append(prop_info)

    return props


def _get_relationship_patterns(driver, rel_type: str, sample: int) -> List[Dict[str, Any]]:
    q = f"""
    MATCH (a)-[r:`{rel_type}`]->(b)
    RETURN DISTINCT labels(a) AS start_labels, type(r) AS type, labels(b) AS end_labels
    LIMIT {sample}
    """
    rows = query_database(driver, q)

    patterns = []
    for row in rows:
        start_labels = row.get("start_labels", [])
        end_labels = row.get("end_labels", [])
        patterns.append(
            {
                "start": ":".join(start_labels) if start_labels else "UNKNOWN",
                "type": row["type"],
                "end": ":".join(end_labels) if end_labels else "UNKNOWN",
            }
        )
    return patterns


def get_structured_schema(
    driver,
    is_enhanced: bool = False,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
    sample: int = 25,
) -> Dict[str, Any]:
    """
    Fallback no-APOC con la stessa API.
    is_enhanced=False/True viene gestito in format_schema.
    """
    node_labels = _get_node_labels(driver)
    rel_types = _get_relationship_types(driver)

    node_props = {}
    for label in node_labels:
        node_props[label] = _get_node_properties(driver, label, sample=sample)

    rel_props = {}
    for rel_type in rel_types:
        rel_props[rel_type] = _get_rel_properties(driver, rel_type, sample=sample)

    relationships = []
    for rel_type in rel_types:
        relationships.extend(_get_relationship_patterns(driver, rel_type, sample=sample))

    structured_schema = {
        "node_props": node_props,
        "rel_props": rel_props,
        "relationships": relationships,
        "metadata": {
            "constraint": [],
            "index": [],
            "source": "no_apoc_fallback",
            "is_enhanced_available": True,
        },
    }

    return structured_schema


def _format_property(prop):
    t = prop["type"]

    # STRING
    if t == "STRING" and prop.get("values"):
        return f'Example: "{_clean_string_values(prop["values"][0])}"'

    # INTEGER
    elif t == "INTEGER":
        if "min" in prop and "max" in prop:
            # rimuove .0 se presente
            min_val = int(prop["min"]) if float(prop["min"]).is_integer() else prop["min"]
            max_val = int(prop["max"]) if float(prop["max"]).is_integer() else prop["max"]
            return f"Min: {min_val}, Max: {max_val}"
        elif prop.get("values"):
            val = prop["values"][0]
            try:
                val = int(val)
            except Exception:
                pass
            return f"Example: {val}"
        return ""

    # FLOAT
    elif t == "FLOAT":
        if "min" in prop and "max" in prop:
            return f"Min: {prop['min']}, Max: {prop['max']}"
        elif prop.get("values"):
            return f"Example: {prop['values'][0]}"
        return ""

    # BOOLEAN
    elif t == "BOOLEAN":
        if prop.get("values"):
            val = prop["values"][0]
            if isinstance(val, str):
                val = val.lower() == "true"
            return f"Example: {val}"
        return ""

    # LIST
    elif t == "LIST":
        parts = []

        if prop.get("item_examples"):
            example = _clean_string_values(prop["item_examples"][0])
            parts.append(f'Item Example: "{example}"')

        if prop.get("min_size") is not None and prop.get("max_size") is not None:
            parts.append(f"Min Size: {prop['min_size']}, Max Size: {prop['max_size']}")

        return " | ".join(parts)

    # fallback
    else:
        if prop.get("values"):
            return f'Example: "{_clean_string_values(prop["values"][0])}"'
        return ""


def _format_properties(property_dict, is_enhanced):
    formatted_props = []

    if is_enhanced:
        for label, props in property_dict.items():
            formatted_props.append(f"- **{label}**")
            for prop in props:
                example = _format_property(prop)
                if example:
                    formatted_props.append(
                        f"  - `{prop['property']}`: {prop['type']} {example}"
                    )
                else:
                    formatted_props.append(
                        f"  - `{prop['property']}`: {prop['type']}"
                    )
    else:
        for label, props in property_dict.items():
            props_str = ", ".join(
                [f"{prop['property']}: {prop['type']}" for prop in props]
            )
            formatted_props.append(f"{label} {{{props_str}}}")

    return formatted_props


def _format_relationships(rels: List[Dict[str, Any]]) -> List[str]:
    out = []
    seen = set()

    for el in rels:
        rel_str = f"(:{el['start']})-[:{el['type']}]->(:{el['end']})"
        if rel_str not in seen:
            seen.add(rel_str)
            out.append(rel_str)

    return out


def format_schema(schema: Dict[str, Any], is_enhanced: bool) -> str:
    formatted_node_props = _format_properties(schema["node_props"], is_enhanced)
    formatted_rel_props = _format_properties(schema["rel_props"], is_enhanced)
    formatted_rels = _format_relationships(schema["relationships"])

    return "\n".join(
        [
            "Node properties:",
            "\n".join(formatted_node_props),
            "The relationships:",
            "\n".join(formatted_rels),
            "Relationship properties:",
            "\n".join(formatted_rel_props),
        ]
    )


def get_schema(
    driver,
    is_enhanced: bool = False,
    database: Optional[str] = None,
    timeout: Optional[float] = None,
    sanitize: bool = False,
    sample: int = 25,
) -> str:
    structured_schema = get_structured_schema(
        driver=driver,
        is_enhanced=is_enhanced,
        database=database,
        timeout=timeout,
        sanitize=sanitize,
        sample=sample,
    )
    return format_schema(structured_schema, is_enhanced=is_enhanced)

In [ ]:
def evaluate_cypher_queries_schema(df, graph, cypher_chain, cypher_prompt, schema, k=1):
    """
    Evaluate cypher queries and update the DataFrame with new evaluation metrics.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least the columns "question" and "cypher".
        graph: Neo4jGraph object used to run queries.
        cypher_chain: LLM chain for generating Cypher queries.
        cypher_prompt: Prompt template for generating Cypher queries.
        k (int): Number of generated Cypher queries to evaluate per question (default is 1).

    Returns:
        pd.DataFrame: The updated DataFrame with new columns:
        "generated_cypher", "true_data", "eval_data", "jaro_winkler", "pass_1", "pass_3", "jaccard".
    """
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    normalized_levenshteins = []
    jaccards = []
    coverages = []
    passes = []
    
    prompt = []

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Get the true data based on the test Cypher statement.
        true_data = graph.query(row["cypher"])
        
        # Generate cypher queries and fetch data.
        example_generated_cyphers = []
        example_eval_datas = []
        prompts = []
        for _ in range(k):
            cypher = cypher_chain.invoke({"question": row["question"]})
            formatted_prompt = cypher_prompt.format_messages(question=row["question"],
                                                             enhanced_schema=schema)
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })
            example_generated_cyphers.append(cypher)
            try:
                example_eval_datas.append(graph.query(cypher))

            except ClientError as e:
                if e.code == "Neo.ClientError.Statement.SyntaxError":
                    print("Cypher SyntaxError:", e)
                    example_eval_datas.append([{"id": "Cypher syntax error"}])

                elif e.code == "Neo.ClientError.Statement.SemanticError":
                    print("Cypher SemanticError:", e)
                    example_eval_datas.append([{"id": "Cypher semantic error"}])

                else:
                    print("Cypher ClientError:", e.code, e)
                    example_eval_datas.append([{"id": "Cypher client error"}])

            except Neo4jError as e:
                print("Neo4j runtime error:", e)
                example_eval_datas.append([{"id": "Neo4j runtime error"}])

            except Exception as e:
                print("Other error:", type(e), e)
                example_eval_datas.append([{"id": "Other error"}])

        # Compute metrics using the first generated cypher/response.
        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])
        
        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        passs = 1 if coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        ) == 1 else 0

        # Append computed values.
        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        
        prompt.append(prompts)

    # Add evaluated results as new columns to the DataFrame.
    df["generated_cypher"] = generated_cyphers
    df["true_data"] = true_datas
    df["eval_data"] = eval_datas
    df["jaro_winkler"] = jaro_winklers
    
    df["jaccard"] = jaccards
    df["coverage"] = coverages
    df["pass"] = passes
    
    df["prompt"] = prompt

    return df

In [ ]:
from neo4j import GraphDatabase
import os

driver = GraphDatabase.driver(
    "bolt://neo4j.het.io",
    auth=("neo4j", "neo4j")
)

simple_schema = get_schema(driver, is_enhanced=False, sample=25)
print(simple_schema)

enhanced_schema = get_schema(driver, is_enhanced=True, sample=25)
print(enhanced_schema)

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{enhanced_schema}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a Cypher query. No pre-amble.",
        ),
        ("human", cypher_template),
    ]
)

cypher_chain = ( RunnablePassthrough.assign(
        enhanced_schema=lambda _: enhanced_schema
    ) | cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)

In [ ]:
df = evaluate_cypher_queries_schema(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_schema.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_schema.pkl")
df.head(n=3)

***
# Examples via RAG

Examples reteieved from a vector store in a RAG fashion are useful to provide context to the LLM. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NLQUESTIONS_PATH = Path("./NLquestions")     
CYPHER_PATH      = Path("./CypherQueries")   

# -----------------------------
# Helpers: mapping source -> cypher file
# -----------------------------
def cypher_file_from_source(meta_source: str) -> Path:
    """
    meta_source: es '3hop/question_3.txt'
    -> CypherQueries/3hop/question_3.txt
    """
    return CYPHER_PATH / Path(meta_source)


# -----------------------------
# RAG: retrieve examples bundle
# -----------------------------
def retrieve_examples_bundle(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source.replace("question", "cypher"))
        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()
            
        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }


def enrich_with_examples(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle(inputs["question"], n_results=3)
    return {
        **inputs,
        "examples": bundle["examples_text"],                 
        "example_ids": bundle["example_ids"],                
        "example_distances": bundle["example_distances"],   
    }

In [ ]:
cypher_template = """Based on the examples below that provide useful
patterns for querying the graph database,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Examples:
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given an input question, convert it to a Cypher query. No pre-amble."),
    ("human", cypher_template),
])

cypher_generate = (
    cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)


def _run_graph_query_safe(graph, cypher: str):
    try:
        return graph.query(cypher)
    except ClientError as e:
        if e.code == "Neo.ClientError.Statement.SyntaxError":
            return [{"id": "Cypher syntax error"}]
        if e.code == "Neo.ClientError.Statement.SemanticError":
            return [{"id": "Cypher semantic error"}]
        return [{"id": f"Cypher client error ({e.code})"}]
    except Neo4jError:
        return [{"id": "Neo4j runtime error"}]
    except Exception:
        return [{"id": "Other error"}]


def evaluate_cypher_queries_RAG(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, k: int = 1) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):

        true_data = _run_graph_query_safe(graph, row["cypher"])

        example_generated_cyphers = []
        example_eval_datas = []
        example_ids_runs = []
        example_dist_runs = []
        prompts = []

        for _ in range(k):
            enriched = enrich_with_examples({"question": row["question"]})

            example_ids_runs.append(enriched["example_ids"])
            example_dist_runs.append(enriched["example_distances"])

            formatted_prompt = cypher_prompt.format_messages(
                question=enriched["question"],
                examples=enriched["examples"]
            )
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })

            cypher = cypher_generate.invoke({
                "question": enriched["question"],
                "examples": enriched["examples"]
            })

            example_generated_cyphers.append(cypher)
            example_eval_datas.append(_run_graph_query_safe(graph, cypher))

        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        passs = 1 if coverage == 1 else 0
        # append
        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        prompt.append(prompts)

        retrieved_example_ids.append(example_ids_runs)
        retrieved_example_distances.append(example_dist_runs)

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverages
    out["pass"] = passes
    out["prompt"] = prompt

    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

In [ ]:
inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")

In [ ]:
df = evaluate_cypher_queries_RAG(test_df, graph, cypher_generate, cypher_prompt)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG.pkl")
df.head(n=3)

***
# Examples + Output

Examples retrieved from a vector store in a RAG fashion are useful to provide context to the LLM. Moreover, their outputs might help the LLM in inferring the useful portion of the database schema structure needed to improve the generated query. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from neo4j.exceptions import ClientError, Neo4jError
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# -----------------------------
# PATHS
# -----------------------------
NLQUESTIONS_PATH = Path("./NLquestions")       
CYPHER_PATH      = Path("./CypherQueries")     
NEO4J_OUTPUTS    = Path("./Neo4jOutputs")     

# -----------------------------
# Mapping meta["source"] -> file paths
# -----------------------------
def cypher_file_from_source(meta_source: str) -> Path:
    # "3hop/question_3.txt" -> "CypherQueries/3hop/cypher_3.txt"
    rel = Path(meta_source)
    name = rel.name.replace("question", "cypher")          # question_3.txt -> cypher_3.txt
    return CYPHER_PATH / rel.parent / name

def output_file_from_source(meta_source: str) -> Path:
    # "3hop/question_3.txt" -> "Neo4jOutputs/3hop/output_3.txt"
    rel = Path(meta_source)
    name = rel.name.replace("question", "output")          # question_3.txt -> output_3.txt
    return NEO4J_OUTPUTS / rel.parent / name

# -----------------------------
# RAG retrieval (examples + ids + distances + outputs)
# -----------------------------
def retrieve_examples_bundle_with_output(question: str, n_results: int = 3):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-large",
        input=[question]
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results.get("distances", [[None] * n_results])[0]

    example_ids = []
    example_distances = []
    prompt_parts = []

    for doc, meta, dist in zip(docs, metas, dists):
        meta_source = meta["source"]  # es: "3hop/question_3.txt"
        example_ids.append(meta_source)
        example_distances.append(dist)

        cypher_file = cypher_file_from_source(meta_source)
        out_file    = output_file_from_source(meta_source)

        if not cypher_file.exists():
            raise FileNotFoundError(f"Cypher file not found for source={meta_source}: {cypher_file}")
        if not out_file.exists():
            raise FileNotFoundError(f"Neo4j output file not found for source={meta_source}: {out_file}")

        with open(cypher_file, "r", encoding="utf-8") as f:
            cypher_query = f.read().strip()

        with open(out_file, "r", encoding="utf-8") as f:
            neo4j_output = f.read().strip()

        prompt_parts.append(f"[Query natural language] {doc}\n")
        prompt_parts.append(f"[Query Cypher] {cypher_query}\n")
        prompt_parts.append(f"[Output Neo4j] {neo4j_output}\n\n")

    return {
        "examples_text": "".join(prompt_parts),
        "example_ids": example_ids,
        "example_distances": example_distances,
    }

def enrich_with_examples_with_output(inputs: dict) -> dict:
    bundle = retrieve_examples_bundle_with_output(inputs["question"], n_results=3)
    return {
        **inputs,
        "examples": bundle["examples_text"],
        "example_ids": bundle["example_ids"],
        "example_distances": bundle["example_distances"],
    }

# -----------------------------
# Prompt + generator
# -----------------------------
cypher_template = """Based on the examples below that provide useful
patterns for querying the graph database (Neo4j outputs are also reported),
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Examples:
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given an input question, convert it to a Cypher query. No pre-amble."),
    ("human", cypher_template),
])

cypher_generate = (
    cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)

# -----------------------------
# Safe Neo4j query
# -----------------------------
def _run_graph_query_safe(graph, cypher: str):
    try:
        return graph.query(cypher)
    except ClientError as e:
        if e.code == "Neo.ClientError.Statement.SyntaxError":
            return [{"id": "Cypher syntax error"}]
        if e.code == "Neo.ClientError.Statement.SemanticError":
            return [{"id": "Cypher semantic error"}]
        return [{"id": f"Cypher client error ({e.code})"}]
    except Neo4jError:
        return [{"id": "Neo4j runtime error"}]
    except Exception:
        return [{"id": "Other error"}]

def evaluate_cypher_queries_RAG_with_output(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, k: int = 1) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        true_data = _run_graph_query_safe(graph, row["cypher"])

        example_generated_cyphers = []
        example_eval_datas = []
        example_ids_runs = []
        example_dist_runs = []
        prompts = []

        for _ in range(k):
            enriched = enrich_with_examples_with_output({"question": row["question"]})
            example_ids_runs.append(enriched["example_ids"])
            example_dist_runs.append(enriched["example_distances"])

            formatted_prompt = cypher_prompt.format_messages(
                question=enriched["question"],
                examples=enriched["examples"]
            )
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })

            cypher = cypher_generate.invoke({
                "question": enriched["question"],
                "examples": enriched["examples"]
            })

            example_generated_cyphers.append(cypher)
            example_eval_datas.append(_run_graph_query_safe(graph, cypher))

        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        passs = 1 if coverage == 1 else 0

        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        prompt.append(prompts)

        retrieved_example_ids.append(example_ids_runs)
        retrieved_example_distances.append(example_dist_runs)

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverages
    out["pass"] = passes
    out["prompt"] = prompt
    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

inputs = {
    "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
}

enriched = enrich_with_examples_with_output(inputs)

prompt_messages = cypher_prompt.format_messages(
    question=enriched["question"],
    examples=enriched["examples"]
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")

In [ ]:
df = evaluate_cypher_queries_RAG_with_output(test_df, graph, cypher_generate, cypher_prompt)
df.head(3)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_RAG+output.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_RAG+output.pkl")
df.head(n=3)

***
# Schema + Examples

Let's combine enhanced schema and examples. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{schema}

Examples:
The following examples provide useful patterns for querying the graph database.
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a Cypher query. No pre-amble.",
        ),
        ("human", cypher_template),
    ]
)

cypher_chain = (
    RunnablePassthrough.assign(
        schema=lambda _: enhanced_schema,
        examples=lambda x: retrieve_examples_bundle(x["question"])
    )
    | cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)

In [ ]:
question = "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"

bundle = retrieve_examples_bundle(question)  # dict
prompt_messages = cypher_prompt.format_messages(
    schema=enhanced_schema,
    examples=bundle["examples_text"],
    question=question
)

print("=== PROMPT ===")
for m in prompt_messages:
    print(f"\n[{m.type.upper()}]\n{m.content}")

In [ ]:
def evaluate_cypher_queries_RAG_schema(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, schema, k: int = 1) -> pd.DataFrame:#3) -> pd.DataFrame: # Uncomment if Pass@3 is needed
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):

        true_data = _run_graph_query_safe(graph, row["cypher"])

        example_generated_cyphers = []
        example_eval_datas = []
        example_ids_runs = []
        example_dist_runs = []
        prompts = []

        for _ in range(k):
            enriched = enrich_with_examples({"question": row["question"]})

            example_ids_runs.append(enriched["example_ids"])
            example_dist_runs.append(enriched["example_distances"])

            formatted_prompt = cypher_prompt.format_messages(
                question=enriched["question"],
                examples=enriched["examples"],
                schema=schema
            )
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })

            cypher = cypher_generate.invoke({
                "question": enriched["question"],
                "examples": enriched["examples"]
            })

            example_generated_cyphers.append(cypher)
            example_eval_datas.append(_run_graph_query_safe(graph, cypher))

        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )
        passs = 1 if coverage == 1 else 0
        # append
        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        prompt.append(prompts)

        retrieved_example_ids.append(example_ids_runs)
        retrieved_example_distances.append(example_dist_runs)

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverages
    out["pass"] = passes
    out["prompt"] = prompt

    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

In [ ]:
df = evaluate_cypher_queries_RAG_schema(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_schema+RAG.pkl")
df.head(n=3)

***
# Schema + Examples + Output

Let's combine enhanced schema, examples, and their outputs. Next, we will use the LangChain expression language to define a prompt sent to the LLM with instructions to translate the natural language to a Cypher statement that retrieves relevant information to answer the question.

In [ ]:
# Generate Cypher statement based on natural language input

cypher_template = """Based on the Neo4j graph schema below,
write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.
{enhanced_schema}

Examples:
The following examples provide useful patterns for querying
the graph database (Neo4j outputs are also reported).
{examples}

Question: {question}
Cypher query:"""  # noqa: E501

cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Given an input question, convert it to a Cypher query. No pre-amble.",
        ),
        ("human", cypher_template),
    ]
)

cypher_chain = (
    RunnablePassthrough.assign(
        enhanced_schema=lambda _: enhanced_schema,
        examples=lambda x: retrieve_examples_bundle_with_output(x["question"])
    )
    | cypher_prompt
    | llm.bind(stop=["\nCypherResult:"], max_tokens=MAX_TOKENS)
    | StrOutputParser()
)


In [ ]:
def evaluate_cypher_queries_RAG_schema_with_output(df: pd.DataFrame, graph, cypher_generate, cypher_prompt, schema, k: int = 1) -> pd.DataFrame:
    generated_cyphers = []
    true_datas = []
    eval_datas = []
    jaro_winklers = []
    jaccards = []
    coverages = []
    passes = []
    prompt = []

    retrieved_example_ids = []
    retrieved_example_distances = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        true_data = _run_graph_query_safe(graph, row["cypher"])

        example_generated_cyphers = []
        example_eval_datas = []
        example_ids_runs = []
        example_dist_runs = []
        prompts = []

        for _ in range(k):
            enriched = enrich_with_examples_with_output({"question": row["question"]})
            example_ids_runs.append(enriched["example_ids"])
            example_dist_runs.append(enriched["example_distances"])

            formatted_prompt = cypher_prompt.format_messages(
                question=enriched["question"],
                examples=enriched["examples"],
                enhanced_schema=schema
            )
            prompts.append({
                "messages": [
                    {"role": msg.type, "content": msg.content}
                    for msg in formatted_prompt
                ],
            })

            cypher = cypher_generate.invoke({
                "question": enriched["question"],
                "examples": enriched["examples"]
            })

            example_generated_cyphers.append(cypher)
            example_eval_datas.append(_run_graph_query_safe(graph, cypher))

        jw_value = get_jw_distance(row["cypher"], example_generated_cyphers[0])

        jaccard = jaccard_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        coverage = coverage_df_sim_pair(
            (row["cypher"], true_data),
            (example_generated_cyphers[0], example_eval_datas[0])
        )

        passs = 1 if coverage == 1 else 0

        generated_cyphers.append(example_generated_cyphers)
        true_datas.append(true_data)
        eval_datas.append(example_eval_datas)
        jaro_winklers.append(jw_value)
        jaccards.append(jaccard)
        coverages.append(coverage)
        passes.append(passs)
        prompt.append(prompts)

        retrieved_example_ids.append(example_ids_runs)
        retrieved_example_distances.append(example_dist_runs)

    out = df.copy()
    out["generated_cypher"] = generated_cyphers
    out["true_data"] = true_datas
    out["eval_data"] = eval_datas
    out["jaro_winkler"] = jaro_winklers
    out["jaccard"] = jaccards
    out["coverage"] = coverages
    out["pass"] = passes
    out["prompt"] = prompt
    out["retrieved_example_ids"] = retrieved_example_ids
    out["retrieved_example_distances"] = retrieved_example_distances

    return out

In [ ]:
df = evaluate_cypher_queries_RAG_schema_with_output(test_df, graph, cypher_chain, cypher_prompt, enhanced_schema)
df.head(n=3)

Let's also save the dataframe so that we avoid running it again and we can inspect it analytically.

In [ ]:
df.to_excel(model_name + "/evaluating_text2cypher_" + model_name + "_Eschema+RAG+output.xlsx", index=False)
df.to_pickle(model_name + "/evaluating_text2cypher_" + model_name + "_Eschema+RAG+output.pkl")
df.head(n=3)